<p>Indic Conformer models (typically from the AI4Bharat NeMo or ESPnet ecosystems) are high-performance ASR architectures. Quantizing them and moving to ONNX can significantly reduce latency and disk footprint, making them suitable for real-time edge deployment.

Since most Indic Conformer models are based on the NeMo framework, the standard path is to export them to ONNX first and then apply Post-Training Quantization (PTQ) using ONNX Runtime.</p>

<p><b>1. Exporting Indic Conformer to ONNX</b>


If you are using the AI4Bharat NeMo version, you should use the built-in export functionality. This handles the complex Conformer sub-modules (Attention, Convolutions) correctly.</p>

In [1]:
import torch
import nemo.collections.asr as nemo_asr

# 1. Load your pretrained model
restore_path = "indic_conformer_model.nemo"
model = nemo_asr.models.EncDecCTCModel.restore_from(restore_path)

# 2. Export to ONNX
# This creates 'indic_conformer.onnx'
model.export("indic_conformer.onnx", check_trace=True)

ModuleNotFoundError: No module named 'nemo'

<p><b>2. Quantizing the ONNX Model</b>


We use onnxruntime.quantization to perform Dynamic Quantization. This is usually the best choice for ASR models like Conformer, as it strikes a balance between performance and accuracy without needing a huge calibration dataset.</p>

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

# Path to your FP32 ONNX model
model_fp32 = "indic_conformer.onnx"
# Path to save the INT8 model
model_quant = "indic_conformer_int8.onnx"

# Quantize weights to INT8
quantize_dynamic(
    model_input=model_fp32,
    model_output=model_quant,
    weight_type=QuantType.QInt8
)

print(f"Quantization complete. Saved to: {model_quant}")

<p><b>3. Inference with the Quantized ONNX Model</b>


Once exported and quantized, you can run inference using onnxruntime. Note that ASR models require a pre-processor (to turn audio into Mel-spectrograms) and a post-processor (to turn logits into text).</p>

In [ ]:
import onnxruntime as ort
import numpy as np

# Load the session
session = ort.InferenceSession("indic_conformer_int8.onnx", providers=['CPUExecutionProvider'])

# Prepare dummy input (audio signal length should match model expectations)
# Usually (Batch, Time) or (Batch, Feat, Time) depending on the export
input_name = session.get_inputs()[0].name
dummy_audio = np.random.randn(1, 16000).astype(np.float32) 

# Run inference
logits = session.run(None, {input_name: dummy_audio})
print("Logits shape:", logits[0].shape)

<p>Key Considerations for Indic Conformer


Decoding Strategy: Indic Conformer supports both CTC and RNN-T decoding. If you are using RNN-T, the ONNX export is more complex because it involves multiple files (Encoder, Decoder, and Joint).

Audio Pre-processing: Don't forget that the ONNX model expects processed features (like log-Mel filterbanks) unless you exported the model with the pre-processor included.

Hardware Acceleration: If you have an NVIDIA GPU, use CUDAExecutionProvider in the InferenceSession for a 5x-10x speedup over the CPU.</p>